# Actividad 11. Predicción de impacto

Además de clasificar el riesgo, el proyecto estima **cuántas personas**
podrían resultar afectadas y **cuánto daño económico** podría registrarse.
Se entrenan dos modelos por objetivo (red neuronal y XGBoost) con la misma
partición temporal.

La variable se modela en escala logarítmica porque el impacto sigue una
distribución muy asimétrica: unos pocos eventos concentran la mayor parte de
las pérdidas. El daño económico es **exploratorio**, porque falta en cerca de
70 % de los eventos.

## 1. Configuración

In [1]:
import json
import math
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

MODELOS = RAIZ / "modelos"
RESULTADOS = RAIZ / "resultados" / "ml"
GRAFICAS = RAIZ / "resultados" / "graficas" / "ml"
TABLAS = RAIZ / "resultados" / "tablas"
for carpeta in (RESULTADOS, GRAFICAS, TABLAS):
    carpeta.mkdir(parents=True, exist_ok=True)

PALETA = {
    "purpura": "#4C1D95",
    "purpura_claro": "#8B5CF6",
    "rosa": "#FA8A9A",
    "amarillo": "#FFB71B",
    "rojo": "#EE3E29",
    "coral": "#FF8A5F",
    "azul": "#1B4576",
    "gris": "#7A7191",
}
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 150,
        "font.family": "DejaVu Sans",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.22,
        "axes.titlesize": 12,
        "axes.labelsize": 10,
    }
)


def guardar(fig, nombre):
    ruta = GRAFICAS / nombre
    fig.savefig(ruta, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"Figura guardada: {ruta.relative_to(RAIZ)}")

In [2]:
metricas = json.loads((MODELOS / "metricas.json").read_text(encoding="utf-8"))

archivos = {
    "afectados": RESULTADOS / "predicciones_afectados_test.csv",
    "dano": RESULTADOS / "predicciones_dano_test.csv",
}
pruebas = {nombre: pd.read_csv(ruta) for nombre, ruta in archivos.items()}

for nombre, datos in pruebas.items():
    print(f"{nombre}: {len(datos):,} eventos de prueba")

afectados: 677 eventos de prueba
dano: 206 eventos de prueba


## 2. Métricas por objetivo y modelo

In [3]:
filas = []
for objetivo, etiqueta in [("regresion_afectados", "Personas afectadas"), ("regresion_dano", "Daño económico")]:
    for modelo, nombre_modelo in [("red_neuronal", "Red neuronal"), ("xgboost", "XGBoost")]:
        for particion, nombre_particion in [("validacion", "Validación 2023"), ("prueba", "Prueba 2024–2025")]:
            m = metricas[objetivo][modelo][particion]
            filas.append(
                {
                    "objetivo": etiqueta,
                    "modelo": nombre_modelo,
                    "particion": nombre_particion,
                    "n": m["n"],
                    "r2_log": m["r2_log"],
                    "mae_log": m["mae_log"],
                    "rmse_log": m["rmse_log"],
                    "medae_log": m["medae_log"],
                    "r2_original": m["r2_original"],
                }
            )
tabla = pd.DataFrame(filas)
display(tabla.round(4))
tabla.to_csv(TABLAS / "actividad11_metricas.csv", index=False)

fig, eje = plt.subplots(figsize=(9, 4.2))
prueba = tabla[tabla["particion"] == "Prueba 2024–2025"]
etiquetas = [f"{fila.objetivo}\n{fila.modelo}" for fila in prueba.itertuples()]
colores = [PALETA["rojo"] if fila.modelo == "Red neuronal" else PALETA["purpura"] for fila in prueba.itertuples()]
eje.bar(etiquetas, prueba["r2_log"], color=colores)
eje.axhline(0, color=PALETA["gris"], linewidth=0.8)
eje.set_ylabel("R² en escala logarítmica")
eje.set_ylim(0, 0.55)
eje.set_title("Capacidad explicativa en prueba")
for i, valor in enumerate(prueba["r2_log"]):
    eje.text(i, valor + 0.01, f"{valor:.3f}", ha="center", fontsize=9)
guardar(fig, "actividad11_r2.png")

,objetivo,modelo,particion,n,r2_log,mae_log,rmse_log,medae_log,r2_original
0,Personas afectadas,Red neuronal,Validación 2023,359,0.3432,2.0608,2.6528,1.7020,0.0021
1,Personas afectadas,Red neuronal,Prueba 2024–2025,677,0.4009,2.0339,2.5873,1.6776,0.0409
2,Personas afectadas,XGBoost,Validación 2023,359,0.3249,2.1086,2.6894,1.6983,0.0004
3,Personas afectadas,XGBoost,Prueba 2024–2025,677,0.3969,2.0609,2.5958,1.7372,0.0415
4,Daño económico,Red neuronal,Validación 2023,94,0.3794,1.5763,2.0923,1.2045,-0.0394
5,Daño económico,Red neuronal,Prueba 2024–2025,206,0.2125,1.5684,2.0179,1.3005,0.0270
6,Daño económico,XGBoost,Validación 2023,94,0.3567,1.6269,2.1302,1.1137,-0.0355
7,Daño económico,XGBoost,Prueba 2024–2025,206,0.2163,1.5952,2.0131,1.3599,0.1906


Figura guardada: resultados\graficas\ml\actividad11_r2.png


## 3. Dispersión observado vs. predicho

La diagonal representa la predicción perfecta. La dispersión es amplia: el
modelo ordena razonablemente el impacto, pero no acierta el valor exacto de
cada evento. Esa incertidumbre se comunica en la app con un intervalo al 80 %.

In [4]:
def dispersion(nombre, etiqueta, archivo):
    datos = pruebas[nombre]
    limite_inferior = min(datos["y_log"].min(), datos["nn_log"].min(), datos["xgb_log"].min())
    limite_superior = max(datos["y_log"].max(), datos["nn_log"].max(), datos["xgb_log"].max())

    fig, ejes = plt.subplots(1, 2, figsize=(11, 4.6), sharex=True, sharey=True)
    for eje, columna, titulo, color in [
        (ejes[0], "nn_log", "Red neuronal", PALETA["rojo"]),
        (ejes[1], "xgb_log", "XGBoost", PALETA["purpura"]),
    ]:
        eje.scatter(datos["y_log"], datos[columna], s=14, alpha=0.45, color=color)
        eje.plot([limite_inferior, limite_superior], [limite_inferior, limite_superior], linestyle="--", color=PALETA["gris"], linewidth=1)
        eje.set_title(f"{etiqueta} · {titulo}")
        eje.set_xlabel("log1p(valor observado)")
        eje.set_ylabel("log1p(valor predicho)")
    guardar(fig, archivo)

dispersion("afectados", "Personas afectadas", "actividad11_afectados_dispersion.png")
dispersion("dano", "Daño económico", "actividad11_dano_dispersion.png")

Figura guardada: resultados\graficas\ml\actividad11_afectados_dispersion.png


Figura guardada: resultados\graficas\ml\actividad11_dano_dispersion.png


## 4. Distribución de residuales

In [5]:
fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
for eje, nombre, etiqueta in [
    (ejes[0], "afectados", "Personas afectadas"),
    (ejes[1], "dano", "Daño económico"),
]:
    datos = pruebas[nombre]
    residual = datos["y_log"] - datos["xgb_log"]
    eje.hist(residual, bins=40, color=PALETA["purpura_claro"], edgecolor="white")
    eje.axvline(0, color=PALETA["rojo"], linestyle="--", linewidth=1.5)
    eje.set_title(f"Residuales de XGBoost · {etiqueta}")
    eje.set_xlabel("Observado − predicho (log)")
    eje.set_ylabel("Eventos")
guardar(fig, "actividad11_residuales.png")

print(f"Sesgo medio afectados: {(pruebas['afectados']['y_log'] - pruebas['afectados']['xgb_log']).mean():.3f}")
print(f"Sesgo medio daño     : {(pruebas['dano']['y_log'] - pruebas['dano']['xgb_log']).mean():.3f}")

Figura guardada: resultados\graficas\ml\actividad11_residuales.png
Sesgo medio afectados: 0.386
Sesgo medio daño     : 0.388


## 5. Cobertura de los intervalos

Los intervalos se construyen con los residuales de validación (percentiles 10
y 90). Un intervalo bien calibrado debería contener cerca de **80 %** de los
valores observados en prueba.

In [6]:
filas = []
for nombre, etiqueta in [("afectados", "Personas afectadas"), ("dano", "Daño económico")]:
    datos = pruebas[nombre]
    dentro = (datos["y_log"] >= datos["xgb_inferior_log"]) & (datos["y_log"] <= datos["xgb_superior_log"])
    ancho = np.expm1(datos["xgb_superior_log"]) - np.expm1(datos["xgb_inferior_log"])
    filas.append(
        {
            "objetivo": etiqueta,
            "n": len(datos),
            "cobertura": float(dentro.mean()),
            "ancho_mediano": float(np.median(ancho)),
            "ancho_medio": float(ancho.mean()),
        }
    )
intervalos = pd.DataFrame(filas)
display(intervalos.round(4))
intervalos.to_csv(TABLAS / "actividad11_intervalos.csv", index=False)

fig, eje = plt.subplots(figsize=(7.2, 4))
eje.bar(intervalos["objetivo"], intervalos["cobertura"] * 100, color=[PALETA["purpura"], PALETA["rojo"]])
eje.axhline(80, color=PALETA["gris"], linestyle="--", linewidth=1.5, label="Objetivo 80 %")
eje.set_ylim(0, 100)
eje.set_ylabel("Cobertura (%)")
eje.set_title("Cobertura de los intervalos al 80 %")
for i, valor in enumerate(intervalos["cobertura"]):
    eje.text(i, valor * 100 + 2, f"{valor * 100:.1f} %", ha="center")
eje.legend(frameon=False)
guardar(fig, "actividad11_intervalos.png")

,objetivo,n,cobertura,ancho_mediano,ancho_medio
0,Personas afectadas,677,0.8095,1.659059e+05,2.320984e+06
1,Daño económico,206,0.7767,1.056680e+08,3.658858e+08


Figura guardada: resultados\graficas\ml\actividad11_intervalos.png


## Conclusión

* En **personas afectadas**, la red neuronal obtiene R² logarítmico de
  **0.401** y XGBoost **0.397**; el error mediano ronda 1.7 unidades log.
* En **daño económico**, ambos modelos explican cerca de **21 %** de la
  variación logarítmica. Es un ejercicio exploratorio: el daño falta en la
  mayoría de los eventos y los valores extremos son difíciles de anticipar.
* Los intervalos cubren aproximadamente **81 %** de los afectados y **78 %**
  del daño, cerca del 80 % nominal.
* Por eso la aplicación muestra una **mediana** y un **intervalo**, no una
  cifra única, y advierte que la estimación no sustituye una alerta oficial.